<div dir="rtl">

# رافِدَين — تدريب Seed-VC على اللهجة العراقية على Google Colab

دفتر Colab واحد متكامل ينجز **الرحلة كاملة** — من جلسة GPU فارغة إلى نموذج
`seed-uvit-whisper-base-f0-44k` مضبوط بدقة على الكلام العراقي، محفوظاً بشكل **دائم** في
Google Drive و HuggingFace Hub:

| المرحلة | ماذا يحدث |
|---|---|
| 1. البيئة | توصيل Drive، فحص GPU/القرص، تثبيت ffmpeg + uv ثم الاعتماديات داخل نواة الجلسة نفسها |
| 2. المستودعات | استنساخ Seed-VC ورافِدَين، واستخراج خط بيانات العراق المدمج (49,194 مقطعاً / 339 ساعة) |
| 3. البيانات | تنزيل 5 مجموعات عراقية نظيفة (منها ≈50 ساعة إضافية من Lahgtna v2 + استوديو الخرّوفا) + جلب مُجزَّأ (ranged HTTP) لساعات YODAS العراقية، ثم تقطيع وترميز Opus 24kHz |
| 4. التدريب | ضبط دقيق يبدأ من نقطة `DiT_seed_v2_uvit_whisper_base_f0_44k_bigvgan_pruned_ft_ema.pth` الرسمية، بتقدم حي ومنحنى خسارة، مع **نسخ احتياطي تلقائي إلى Drive أثناء التدريب** |
| 5. التحقق والتصدير | استدلال حقيقي بالنقطة الناتجة (استماع داخل الدفتر)، ثم حزمة تصدير (أوزان + إعداد + تقرير + سجلات) |
| 6. الحفظ الدائم | نسخة في **Google Drive** + أرشيف tar.gz + **رفع إلى HuggingFace Hub** (دائم فعلاً) |

> هذا الدفتر مبني مباشرة على خط التدريب الرسمي للمستودع (`seed_vc_iraqi_colab.sh`) — نفس المراحل
> ونفس الإصدارات المثبتة بالضبط — لكنه مقسّم إلى خلايا قابلة للتشغيل والمراقبة والاستئناف.
> كل خلية آمنة لإعادة التشغيل: المراحل تتخطى ما أُنجز منها (علامات `.done` وفحوص وجود)
> وتستأنف ما بقي. صُمم خصيصاً لطبيعة Colab: جلسات منقطعة، قرص محلي مؤقت، وDrive دائم بطيء.

**ضمانات الدفتر**: صحة المسار كاملة (نفس إصدارات الخط الرسمي المُجرَّب)، استئناف بعد أي انقطاع،
تحقق آلي قبل التصدير، وحفظ ثلاثي الطبقات لا يضيع منه شيء. جودة المخرج النهائية دالةٌ على حجم
البيانات (200 ساعة افتراضياً من أصل 339)، وعدد الخطوات (3000)، وذاكرة GPU — وكلها مقابض قابلة
للرفع في الخلية 2 إن أردت المزيد.

**المصدر**: [github.com/Alqudimi/rafidain](https://github.com/Alqudimi/rafidain) ·
النموذج: [Plachta/Seed-VC](https://huggingface.co/Plachta/Seed-VC)‏ (`seed-uvit-whisper-base-f0-44k`)

</div>

<div dir="rtl">

## ١) تجهيز Colab (قبل تشغيل الخلايا)

### أ. اختيار الـ GPU
من القائمة: **Runtime ← Change runtime type ← Hardware accelerator: GPU** ثم Save.

| GPU | كيف تحصل عليه | الذاكرة | حجم الدفعة المقترح | ملاحظات |
|---|---|---|---|---|
| **T4** | المجاني الافتراضي | 16GB | 1 | يعمل كاملاً — 3000 خطوة في نحو 4–7 ساعات (النموذج الأكبر أدق) |
| **L4** | Colab Pro | 22.5GB | 1–2 | أسرع من T4 بنحو الضعف |
| **A100 40GB** | Colab Pro / Pro+ | 40GB | 4 | فائض عن حاجة الضبط الدقيق |

الدفتر يضبط حجم الدفعة تلقائياً حسب الذاكرة المكتشفة (الخلية 3) — لا تحتاج تعديل شيء. (نموذج base f0 أكبر: الدفعات أصغر عن قصد.)

### ب. حدود جلسة Colab — و كيف يتعامل معها الدفتر (بصراحة)
- الجلسة المجانية تعيش **حتى ~12 ساعة**، وتُقطع بعد **~90 دقيقة خمول** (لا نشاط في النواة).
- **الحلول المطبقة هنا**: خلية «انتظار الاكتمال» (16) تبقي النواة نشطة أثناء التدريب؛
  **مراقب النسخ** (الخلية 14) ينسخ كل نقطة تفتيش جديدة إلى Drive فور ظهورها (كل 250 خطوة) —
  أي أن أقصى خسارة عند أي انقطاع = آخر 250 خطوة فقط.
- بعد أي انقطاع: **Runtime ← Run all** — الخلايا تُعيد بناء البيئة، والخلية 13 تستعيد النقاط
  من Drive، والتدريب يُكمل من آخر نقطة حفظ تلقائياً. البيانات الخام تُعاد جلبها فقط إذا فُقد
  القرص المحلي (كل التنزيلات قابلة للاستئناف).

### ج. Google Drive — طبقة التخزين الدائم
- تحتاج **~2GB** من حصتك (آخر نقطتين ≈ 0.4GB لكل نقطة + النموذج النهائي + السجلات + حزمة
  التصدير). حصة Drive المجانية (15GB) تكفي تماماً.
- بيانات التدريب الثقيلة (~30GB لـ 200 ساعة) تبقى على **القرص المحلي السريع** `/content` —
  عمداً: Drive بطيء جداً لآلاف الملفات الصغيرة، والبيانات قابلة لإعادة الجلب في أي وقت.

### د. قرص Colab المحلي
| الإعداد | تنزيل YODAS الخام | الإجمالي التقريبي (بيانات + بيئة + نقاط) |
|---|---|---|
| 50 ساعة | ≈ 3.1GB | ≈ 12GB |
| **200 ساعة (الافتراضي)** | **≈ 12.7GB** | **≈ 30GB** |
| 334 ساعة (الكتالوج كله) | ≈ 22.4GB | ≈ 42GB |

القرص المحلي في Colab عادة 100GB+ — كل الإعدادات أعلاه تدخل بارتياح.

### هـ. فتح هذا الدفتر
نزّل ملف `rafidain_colab_training.ipynb` وارفعه إلى [colab.research.google.com](https://colab.research.google.com)
(أو ضعه في Drive وافتحه بـ *Open with ← Colab*)، فعّل GPU كما في (أ)، ثم شغّل الخلايا من الأعلى للأسفل.

</div>

<div dir="rtl">

## ٢) خريطة الدفتر — شغّل بالترتيب من الأعلى

| # | الخلية | الوصف | المدة التقريبية |
|---|---|---|---|
| 1 | توصيل Google Drive | طبقة التخزين الدائم (تخطَّها إن أردت العمل بلا Drive) | ثوانٍ |
| 2 | الإعدادات المركزية | كل مقابض التحكم + الأدوات — تُحفظ على Drive | ثوانٍ |
| 3 | فحص البيئة | GPU/قرص/أدوات + تحديد حجم الدفعة تلقائياً | ثوانٍ |
| 4 | تبعيات النظام | ffmpeg + git + uv (أداة تسريع التثبيت) | أقل من دقيقة |
| 5 | الاعتماديات | تثبيت داخل نواة الجلسة عبر uv: PyTorch 2.4.0 cu121 + كل الحزم | 3–8 دقائق |
| 6 | المستودعات | استنساخ Seed-VC + رافِدَين | أقل من دقيقة |
| 7 | خط البيانات | استخراج خط بيانات العراق المدمج + إحصاءات الكتالوج | ثوانٍ |
| 8 | المجموعات النظيفة | youssef + shiekh + safaa (نحو 3GB) | 2–10 دقائق |
| 9 | بيانات YODAS | جلب مُجزَّأ بحسب الساعات المطلوبة (12.7GB لـ 200 ساعة) | 15–60 دقيقة |
| 10 | بياناتك الخاصة | (اختياري) مجلد لملفاتك الصوتية يُدمج في التدريب | حسبك |
| 11 | التقطيع والترميز | تقطيع 1–29.5 ثانية وترميز Opus 24kHz أحادي | 5–20 دقيقة |
| 12 | نقاط النموذج | تنزيل DiT base f0/Whisper/BigVGAN/RMVPE/CAMPPlus مسبقاً (نحو 3.5GB) | 3–12 دقيقة |
| 13 | استعادة من Drive | (بعد أي انقطاع) إعادة نقاط التدريب من Drive — ثم يستأنف التدريب | ثوانٍ |
| 14 | التدريب | **داخل نواة الدفتر مباشرة** + مرآة نسخ حية إلى Drive | مدة التدريب |
| 15 | مراقبة التدريب | تقدم + ETA + منحنى الخسارة (تعمل أثناء التدريب في خلية أخرى) | حسب الرغبة |
| 16 | مراقبة حية | تحديث كل 30 ثانية + مزامنة Drive (تشغَّل بجانب الخلية 14) | مدة التدريب |
| 17 | التحقق | استدلال حقيقي + استماع داخل الدفتر | 1–3 دقائق |
| 18 | التصدير | حزمة أوزان + إعداد + تقرير → ونسخها إلى Drive | ثوانٍ |
| 19 | أرشيف tar.gz | ملف واحد في Drive جاهز للتنزيل | ثوانٍ |
| 20 | HuggingFace Hub | **التخزين السحابي الدائم** — النموذج في حسابك للأبد | 1–5 دقائق |
| 21 | تنظيف المساحة | (اختيارية) حذف المصادر الخام القابلة لإعادة الجلب | ثوانٍ |
| 22 | النشر والاستخدام | أوامر جاهزة: استدلال فوري + خدمة رافِدَين على نفس الجلسة | اختيارية |

**قواعد ذهبية**
- كل الخلايا **آمنة لإعادة التشغيل** — لا شيء يُعاد تنفيذه إن كان منجزاً (علامات `.done` وفحوص وجود).
- إذا انقطعت الجلسة كلها: افتح الدفتر وشغّل **Run all** — البيئة تُبنى، والخلية 13 تستعيد نقاط
  التدريب من Drive، والتدريب يُكمل من آخر نقطة حفظ.
- التدريب يعمل **داخل نواة الدفتر** (خلية 14): التقدم يظهر حياً في الخلية، والخلية 15/16 تراقب السجل في أي وقت.
- **لا تعتمد على القرص المحلي `/content` للديمومة** — كل ما يهم يُنسخ تلقائياً إلى Drive (الخلية 14).

</div>

In [ ]:
# ============================================================ #
#  الخلية ١ — توصيل Google Drive (طبقة التخزين الدائم)         #
#  هنا تُحفظ: الإعدادات + نقاط التدريب + حزم النموذج النهائية   #
#  (يمكن تخطيها — الدفتر يعمل بلا Drive لكن لا شيء ينجو من      #
#   إعادة تعيين الجلسة إلا مرفوعاً إلى HuggingFace لاحقاً)      #
# ============================================================ #
from pathlib import Path
import shutil

DRIVE_MNT = Path("/content/drive")
DRIVE_OK, DRIVE_ROOT = False, None

try:
    from google.colab import drive          # متاح على Colab فقط
    drive.mount(str(DRIVE_MNT))
    DRIVE_OK = (DRIVE_MNT / "MyDrive").is_dir()
except ModuleNotFoundError:
    print("[Drive] لست على Colab — تخطّي توصيل Drive (العمل محلي بالكامل).")
except Exception as e:
    print(f"[Drive] تعذر التوصيل: {e} — تابع بلا Drive أو أعد هذه الخلية.")

if DRIVE_OK:
    DRIVE_ROOT = DRIVE_MNT / "MyDrive" / "rafidain_seedvc"
    for sub in ("runs", "exports"):
        (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
    try:
        du = shutil.disk_usage(DRIVE_ROOT)
        print(f"[Drive] موصول — التخزين الدائم: {DRIVE_ROOT}")
        print(f"[Drive] الحصة: حر {du.free/1e9:.1f}GB من {du.total/1e9:.1f}GB (النسخ الاحتياطية تحتاج ~2GB)")
    except Exception:
        print(f"[Drive] موصول — التخزين الدائم: {DRIVE_ROOT}")
    print("[Drive] ما يُحفظ هنا: nb_config.json + runs/<اسم-التشغيل>/ + exports/<اسم-التشغيل>/")
else:
    print("""
[تحذير] العمل بلا Drive: كل ما يبقى على /content يضيع عند انقطاع الجلسة.
         يبقى خيارك الدائم الوحيد حينها الرفع إلى HuggingFace Hub (الخلية 20).
         لتوصيل Drive لاحقاً: شغّل هذه الخلية ثم أعد تشغيل الخلية 2.""")

In [ ]:
# ============================================================ #
#  الخلية ٢ — الإعدادات المركزية والأدوات المساعدة              #
#  كل مقابض التحكم في مكان واحد، وتُحفظ على Drive إن وُجد       #
#  (إعادة تشغيل النواة/الجلسة لا تفقدها — وتُعاد عند الاستئناف) #
# ============================================================ #
import os, sys, json, subprocess, shutil, time, re
from pathlib import Path

# ---------- 1) حالة Drive من الخلية 1 (آمنة بعد إعادة تشغيل النواة) ----------
DRIVE_OK   = globals().get("DRIVE_OK", False)
DRIVE_ROOT = globals().get("DRIVE_ROOT", None)

# ---------- 2) جذر العمل: القرص المحلي السريع دائماً (Drive بطيء للبيانات!) ----------
# على Colab: /content هو القرص المحلي. IRAQI_DATA_ROOT = مسار مخصص (نفس دلالته في سكربتات المستودع).
ROOT = Path(os.environ.get("IRAQI_DATA_ROOT", "").strip() or "/content/iraqi_seedvc")
ROOT.mkdir(parents=True, exist_ok=True)

# الإعدادات تُحفظ على Drive إن كان موصولاً — فتنجو من إعادة تعيين الجلسة
CFG_DIR  = DRIVE_ROOT if (DRIVE_OK and DRIVE_ROOT) else ROOT
CFG_PATH = CFG_DIR / "nb_config.json"

# ---------- 3) المقابض الافتراضية ----------
DEFAULTS = dict(
    # البيانات
    YODAS_HOURS    = 200,    # ساعات YODAS العراقية (0 = تعطيل، "all" = الكل 334 ساعة، أو أي رقم)
    SKIP_SMALL     = False,  # تخطي المجموعات النظيفة الخمس (youssef/shiekh/safaa/lahgtna/kharrufa)
    # التدريب
    RUN_NAME       = "iraqi_v1",  # اسم التشغيل (مجلد النقاط والحزمة)
    STEPS          = 3000,   # عدد خطوات الضبط الدقيق (نموذج base أكبر: 3000-6000 للهجة عموماً)
    BATCH          = "auto", # حجم الدفعة: "auto" = حسب ذاكرة GPU، أو رقم 1/2/4
    SAVE_EVERY     = 500,    # حفظ نقطة تفتيش كل N خطوة (نقطة النموذج الكبير ≈ 0.6GB — تُنسخ تلقائياً إلى Drive)
    TRAIN_WORKERS  = 2,      # عمال تحميل بيانات التدريب
    # التنزيل
    FETCH_WORKERS  = 16,     # تنزيلات YODAS المتوازية (ارفعها إلى 32 إذا كانت الشبكة سريعة)
    PREP_WORKERS   = 0,      # عمال التقطيع والترميز (0 = تلقائي: حتى 16 نواة)
    HF_ENDPOINT    = "",     # مرآة HuggingFace إن لزم (مثال: https://hf-mirror.com) — اتركها فارغة عادة
    # النسخ الاحتياطي إلى Google Drive
    DRIVE_BACKUP   = True,   # مراقب النسخ التلقائي أثناء التدريب (الخلية 14)
    DRIVE_CKPT_KEEP = 2,     # كم نقطة تفتيش نُبقي على Drive (نقطة ≈ 0.4GB)
    # التخزين الدائم (HuggingFace Hub)
    HF_TOKEN       = "",     # رمز من hf.co/settings/tokens بصلاحية Write — للرفع الدائم (الخلية 20)
    HF_REPO_ID     = "",     # فارغ = <اسم-حسابك>/rafidain-seedvc-iraqi تلقائياً
    HF_PRIVATE     = True,   # مستودع خاص لا يراه أحد غيرك
    # تنظيف
    CLEAN_RAW_AFTER = False, # حذف المصادر الخام بعد تجهيز بيانات التدريب (يوفر مساحة — الخلية 21)
)

# ---------- 4) إعادة تحميل الإعدادات المحفوظة (استئناف بعد أي انقطاع) ----------
CFG = dict(DEFAULTS)
FORCE_RESET = False   # اجعلها True مؤقتاً لإعادة كل الإعدادات إلى الافتراضية
if CFG_PATH.exists() and not FORCE_RESET:
    try:
        saved = json.loads(CFG_PATH.read_text(encoding="utf-8"))
        CFG.update({k: saved[k] for k in DEFAULTS if k in saved})
        print(f"[إعدادات] أُعيد تحميلها من {CFG_PATH} (عدّل ثم أعد التشغيل، أو FORCE_RESET=True للبدء من جديد)")
    except Exception as e:
        print(f"[إعدادات] تعذر قراءة المحفوظة ({e}) — نستخدم الافتراضية")
# ← عدّل أي قيمة هنا ثم أعد تشغيل الخلية، مثال:
#    CFG["YODAS_HOURS"] = 50
#    CFG["STEPS"] = 1000
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---------- 5) المسارات الثابتة ----------
SEEDVC   = ROOT / "seed-vc"                    # مستودع Seed-VC (المدرب الأصلي)
RAFIDAIN = ROOT / "rafidain"                   # مستودع رافِدَين (الخدمة + خط البيانات)
PIPE     = ROOT / "iraqi_pipeline.py"          # خط بيانات العراق — يُستخرج في الخلية 7
DATA_DIR = ROOT / "data" / "iraqi_finetune"    # بيانات التدريب المجهزة (Opus 24kHz)
RUN_DIR  = SEEDVC / "runs" / CFG["RUN_NAME"]   # نقاط التدريب والسجلات (محلي سريع)
EXPORTS  = ROOT / "exports"                    # حزم النموذج النهائية (محلي)
# (لا حاجة لبيئة منفصلة: كل الاعتماديات تُثبَّت داخل نواة الجلسة نفسها — الخلية 5)

# مراياها على Drive (تُنشأ عند الحاجة — لا الآن إن لم يكن موصولاً)
def drive_run_dir():
    return (DRIVE_ROOT / "runs" / CFG["RUN_NAME"]) if (DRIVE_OK and DRIVE_ROOT) else None
def drive_export_base():
    return (DRIVE_ROOT / "exports") if (DRIVE_OK and DRIVE_ROOT) else None

for p in (ROOT, EXPORTS, RUN_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ---------- 6) بيئة الأوامر وأداة تشغيل موحدة ----------
ENV = dict(os.environ)
ENV["PYTHONUNBUFFERED"] = "1"
ENV["HF_HUB_DISABLE_XET"] = "1"           # تنزيل HTTP قابل للاستئناف — Xet يقيّد IPs Colab (خيار الخط الرسمي)
ENV["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
if CFG["HF_ENDPOINT"]:
    ENV["HF_ENDPOINT"] = CFG["HF_ENDPOINT"]

def sh(cmd, cwd=None, env=None, check=True, echo=True):
    # تشغيل أمر وطباعة مخرجاته لحظياً داخل الخلية — يعيد رمز الخروج
    cmd = [str(c) for c in cmd]
    if echo:
        print("$ " + " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None,
                            env=env or ENV, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"فشل الأمر (رمز {rc}): {' '.join(cmd)}")
    return rc

# ---------- 7) ملخص ----------
print("=" * 64)
print(f"جذر العمل (محلي سريع) : {ROOT}")
print(f"التخزين الدائم (Drive) : {DRIVE_ROOT if DRIVE_OK else 'غير موصول — اعتمد على الخلية 20 لاحقاً'}")
print("=" * 64)
print(f"  • ساعات YODAS        : {CFG['YODAS_HOURS']}")
print(f"  • التدريب            : {CFG['STEPS']} خطوة، حفظ كل {CFG['SAVE_EVERY']}, دفعة {CFG['BATCH']}")
print(f"  • اسم التشغيل        : {CFG['RUN_NAME']}")
print(f"  • نسخ Drive التلقائي : {'مفعّل (آخر %d نقاط)' % CFG['DRIVE_CKPT_KEEP'] if (DRIVE_OK and CFG['DRIVE_BACKUP']) else ('غير متاح (Drive غير موصول)' if not DRIVE_OK else 'معطّل')}")
print(f"  • رمز HuggingFace    : {'مضبوط (سيُرفع للأبد في الخلية 20)' if CFG['HF_TOKEN'] else 'غير مضبوط (اختياري — الرفع الدائم)'}")
print(f"\n[المسارات]  البيانات: {DATA_DIR}")
print(f"            التدريب : {RUN_DIR}")
print(f"            التصدير : {EXPORTS}")
print(f"\nملاحظة: الرمز السري (إن وضعته) يُحفظ في {CFG_PATH} — على Drive الخاص بك فقط.")

In [ ]:
# ============================================================ #
#  الخلية ٣ — فحص البيئة: GPU / قرص / أدوات                    #
#  (يحدد حجم الدفعة تلقائياً حسب ذاكرة GPU المكتشفة)           #
# ============================================================ #
import subprocess, os, shutil, re, json, sys
from pathlib import Path

def gpu_query():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
        return [l.strip() for l in out if l.strip()]
    except Exception:
        return []

gpus = gpu_query()
if gpus:
    for i, g in enumerate(gpus):
        parts = [p.strip() for p in g.split(",")]
        name, vram_mib, driver = parts[0], float(parts[1]), parts[2]
        print(f"[GPU {i}] {name} — ذاكرة {vram_mib/1024:.1f}GB — مشغل {driver}")
    if any("T4" in g for g in gpus):
        print("        [T4] الجلسة المجانية — 3000 خطوة تستغرق نحو 2-4 ساعات. اصبر أو ترقَّ إلى L4.")
else:
    print("[GPU] لا يوجد GPU! من القائمة: Runtime <- Change runtime type <- GPU ثم أعد تشغيل")
    print("        الجلسة (Runtime <- Run all) — تجهيز البيانات يعمل بدون GPU، التدريب لا.")

def auto_batch(vram_gb):
    # نموذج base f0 (44.1kHz، نحو 78M باراميتر) أكبر من tiny — دفعة 1 هي الأمتن على 16GB
    if vram_gb >= 40: return 4
    if vram_gb >= 24: return 2
    return 1

if gpus:
    vram_gb = float([p.strip() for p in gpus[0].split(",")][1]) / 1024
    CFG["_BATCH"] = auto_batch(vram_gb)
else:
    vram_gb = 0.0
    CFG["_BATCH"] = 2
if str(CFG["BATCH"]).lower() == "auto":
    BATCH = CFG["_BATCH"]
    print(f"[دفعة] تلقائي => BATCH = {BATCH} (حسب {vram_gb:.0f}GB ذاكرة)")
else:
    BATCH = int(CFG["BATCH"])
    print(f"[دفعة] يدوي  => BATCH = {BATCH}")
CFG["_BATCH"] = BATCH
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- المعالج والذاكرة ----
cpu = os.cpu_count() or 4
try:
    mem_kb = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text()).group(1))
    print(f"[نظام] {cpu} نواة معالج — ذاكرة RAM ≈ {mem_kb/1e6:.1f}GB — بايثون النواة {sys.version_info.major}.{sys.version_info.minor}")
except Exception:
    print(f"[نظام] {cpu} نواة معالج — بايثون النواة {sys.version_info.major}.{sys.version_info.minor}")

# ---- القرص المحلي (حجم البيانات الضخمة) ----
du = shutil.disk_usage(ROOT)
print(f"[قرص محلي] المتاح على {ROOT}: حر {du.free/1e9:.1f}GB من {du.total/1e9:.1f}GB")
need = {"50": 12, "200": 30, "334": 42}.get(str(CFG["YODAS_HOURS"]), 30)
if du.free / 1e9 < need + 5:
    print(f"          [تحذير] إعدادك يحتاج ~{need}GB — قلّل YODAS_HOURS أو شغّل خلية التنظيف (21)")
else:
    print(f"          [جيد] يكفي لإعدادك الحالي (~{need}GB مطلوبة تقريباً)")

# ---- Drive ----
if DRIVE_OK and DRIVE_ROOT:
    try:
        ddu = shutil.disk_usage(DRIVE_ROOT)
        print(f"[Drive] الحصة: حر {ddu.free/1e9:.1f}GB من {ddu.total/1e9:.1f}GB — النسخ الاحتياطية تحتاج ~2GB")
    except Exception:
        print("[Drive] موصول (تعذر قراءة الحصة)")

# ---- الأدوات ----
for tool in ("ffmpeg", "ffprobe", "git", "curl"):
    print(f"[أداة] {tool:<8}: {'متوفر' if shutil.which(tool) else 'غير متوفر — تثبّته الخلية 4'}")
print(f"[أداة] uv       : {'متوفر' if shutil.which('uv') else 'غير متوفر — تثبّته الخلية 4 (اختياري: يسرّع التثبيت داخل النواة)'}")

print("\n[فحص مبدئي] أكمل إلى الخلية 4 (التبعيات).")

In [ ]:
# ============================================================ #
#  الخلية ٤ — تبعيات النظام (ffmpeg / git) + مدير الحزم uv     #
#  (Colab يعمل بصلاحيات root عادة — بلا sudo)                  #
# ============================================================ #
import shutil, os, subprocess
from pathlib import Path

def have(x):
    return shutil.which(x) is not None

missing = [p for p in ("ffmpeg", "git") if not have(p)]
if missing:
    print(f"تثبيت حزم النظام: {', '.join(missing)} ...")
    sh(["apt-get", "update", "-qq"], check=False)
    sh(["apt-get", "install", "-y", "-qq", *missing])
for tool in ("ffmpeg", "git", "curl"):
    print(f"  {tool:<8}: {'متوفر' if have(tool) else 'غير متوفر!'}")

# ---- uv: مُسرّع تثبيت بايثون (نفس مسار سكربتات المستودع الرسمية) ----
# ملاحظة: uv أداة نظام فقط هنا — تثبيت الحزم نفسه يتم داخل نواة الجلسة (الخلية 5)
UV = shutil.which("uv")
if UV is None:
    print("\nتثبيت uv (مرة واحدة لكل جلسة) ...")
    # ننزّل المثبّت ونسجّله عبر sh() — بلا أوامر قشرية غير مرئية
    inst = ROOT / "uv_installer.sh"
    sh(["curl", "-LsSf", "-o", str(inst), "https://astral.sh/uv/install.sh"], check=False)
    if inst.exists():
        sh(["bash", str(inst)], check=False)
        inst.unlink(missing_ok=True)
    for cand in (Path.home() / ".local" / "bin" / "uv", Path.home() / ".cargo" / "bin" / "uv"):
        if cand.exists():
            UV = str(cand)
            break
    if UV:
        os.environ["PATH"] = str(Path(UV).parent) + os.pathsep + os.environ["PATH"]
        ENV["PATH"] = os.environ["PATH"]
if UV:
    r = subprocess.run([UV, "--version"], capture_output=True, text=True)
    print(f"\nuv جاهز: {r.stdout.strip() or r.stderr.strip()}")
    print("سيُستخدم في الخلية 5 لتثبيت الحزم داخل نواة الجلسة نفسها (لا بيئة منفصلة).")
else:
    print("\nuv غير متوفر — سنعتمد pip العادي داخل النواة في الخلية 5 (يعمل أيضاً).")

In [ ]:
# ============================================================ #
#  الخلية ٥ — تثبيت الاعتماديات داخل نواة الجلسة مباشرة        #
#  (uv pip install --python sys.executable — بلا بيئة منفصلة:  #
#   التدريب والاستدلال يعملان بالاستيراد المباشر في النواة،    #
#   وسرعة uv تحل تعارضات الإصدارات في دقائق لا ساعات)          #
# ============================================================ #
import subprocess, importlib, shutil, sys
from pathlib import Path

UV = globals().get("UV") or shutil.which("uv")   # تحصين: يعمل حتى بعد إعادة تشغيل النواة
PY = sys.executable                               # بايثون النواة نفسها — هنا يُثبَّت كل شيء

def have(mods):
    """فحص سريع: هل كل الوحدات قابلة للاستيراد داخل النواة الآن؟"""
    miss = []
    for m in mods:
        try:
            importlib.import_module(m)
        except Exception:
            miss.append(m)
    return miss

def install(pkgs, note):
    """تثبيت قائمة حزم داخل نواة الجلسة — عبر uv إن توفر (أسرع بعشرات المرات)، وإلا pip."""
    print(f"تثبيت {note} داخل نواة الجلسة ({len(pkgs)} حزمة) ...")
    if UV:
        rc = sh([UV, "pip", "install", "--python", PY, *pkgs], check=False)
        if rc != 0:
            sh([UV, "pip", "install", "--python", PY, *pkgs])
    else:
        rc = sh([PY, "-m", "pip", "install", "-q", *pkgs], check=False)
        if rc != 0:
            sh([PY, "-m", "pip", "install", "-q", *pkgs])

# ---- 1) PyTorch 2.4.0 cu121 (يُتخطى إن كان مثبتاً داخل النواة) ----
try:
    import torch as _t
    tv = (_t.__version__ or "")
except Exception:
    tv = ""
if not tv.startswith("2.4."):
    print("تثبيت PyTorch 2.4.0 + torchaudio (cu121) — نحو 2.5GB، عدة دقائق حسب الشبكة ...")
    trio_pin = ["torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0",
                "--index-url", "https://download.pytorch.org/whl/cu121"]
    trio_any = ["torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"]
    if UV:
        rc = sh([UV, "pip", "install", "--python", PY, *trio_pin], check=False)
        if rc != 0:
            sh([UV, "pip", "install", "--python", PY, *trio_any])
    else:
        rc = sh([PY, "-m", "pip", "install", "-q", *trio_pin], check=False)
        if rc != 0:
            sh([PY, "-m", "pip", "install", "-q", *trio_any])
else:
    print(f"PyTorch {tv} موجود أصلاً داخل النواة — تخطّي التثبيت.")

# ---- 2) بقية الاعتماديات (نفس الإصدارات التي نجح بها الخط الرسمي) ----
NEED = ["torch", "torchaudio", "librosa", "soundfile", "munch", "einops",
        "transformers", "pyarrow", "yaml", "matplotlib"]
miss = have(NEED)
if miss:
    print(f"وحدات ناقصة داخل النواة: {', '.join(miss)}")
    pins = ["numpy==1.26.4", "scipy==1.13.1", "librosa==0.10.2", "soundfile==0.12.1",
            "munch==4.0.0", "einops==0.8.0", "transformers==4.46.3", "pyyaml", "tqdm",
            "requests", "pyarrow==17.0.0", "huggingface_hub", "matplotlib"]
    install(pins, "الاعتماديات (numpy/scipy/librosa/transformers/...)")
else:
    print("الاعتماديات موجودة أصلاً داخل النواة — تخطّي التثبيت.")

# ---- 3) تحقق نهائي داخل النواة نفسها ----
import torch, torchaudio
print(f"torch {torch.__version__} | cuda: {torch.cuda.is_available()} | داخل النواة: {PY}")
assert torch.__version__.startswith("2.4."), "فشل تثبيت PyTorch 2.4.x داخل النواة"
print("\n[جاهز] كل الاعتماديات داخل نواة الجلسة — التدريب سيعمل بالاستيراد المباشر (بلا أي عملية ابن).")

In [ ]:
# ============================================================ #
#  الخلية ٦ — استنساخ المستودعات: Seed-VC + رافِدَين            #
#  (على القرص المحلي — يُحدثان إن وُجدا بدل إعادة التنزيل)      #
# ============================================================ #
from pathlib import Path

def git_clone(url, dst):
    if (dst / ".git").exists():
        print(f"تحديث {dst.name} ...")
        sh(["git", "-C", str(dst), "pull", "--ff-only"], check=False)
    else:
        print(f"استنساخ {dst.name} من {url} ...")
        sh(["git", "clone", "--depth", "1", url, str(dst)])

git_clone("https://github.com/Plachtaa/seed-vc.git", SEEDVC)
git_clone("https://github.com/Alqudimi/rafidain.git", RAFIDAIN)

# تحقق من الملفات الحرجة
for f in ("train.py", "inference.py", "configs/presets/config_dit_mel_seed_uvit_whisper_base_f0_44k.yml"):
    assert (SEEDVC / f).exists(), f"ملف مفقود في seed-vc: {f}"
assert (RAFIDAIN / "seed_vc_iraqi_ubuntu.sh").exists(), "سكربت خط البيانات مفقود في rafidain"
print(f"\n[جاهز] seed-vc   : {SEEDVC}")
print(f"[جاهز] rafidain  : {RAFIDAIN}")

In [ ]:
# ============================================================ #
#  الخلية ٧ — استخراج خط بيانات العراق المدمج                  #
#  (الكتالوج العراقي كله مضمّن داخل سكربت المستودع نفسه:        #
#   49,194 مقطعاً / 339 ساعة / خريطة إزاحات بايت-دقيقة          #
#   داخل أرشيفات espnet/yodas3 — نستخرجه هنا كوحدة مستقلة)     #
# ============================================================ #
import re, gzip, base64, json, ast, urllib.request
from pathlib import Path

SRC_SH = RAFIDAIN / "seed_vc_iraqi_ubuntu.sh"
if not SRC_SH.exists():   # احتياط: المستودع غير منسوخ — ننزّل السكربت مباشرة من GitHub
    print("تنزيل السكربت من GitHub مباشرة ...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Alqudimi/rafidain/main/seed_vc_iraqi_ubuntu.sh", SRC_SH)

text = SRC_SH.read_text(encoding="utf-8")
m = re.search(r"cat > \"\$ROOT/iraqi_pipeline\.py\" <<'PYEOF'\n(.*?)\nPYEOF\n", text, re.S)
assert m, "لم يُعثر على خط البيانات المدمج داخل السكربت!"
code = m.group(1)
ast.parse(code)   # تحقق من سلامة الكود المستخرج قبل الكتابة
PIPE.write_text(code, encoding="utf-8")
print(f"[خط البيانات] استُخرج إلى: {PIPE}  ({len(code):,} بايت)")

# ---- إحصاءات الكتالوج العراقي المدمج (بدون أي تنزيل) ----
def _grab(name):
    blob = re.search(name + r' = r"""(.*?)"""', code, re.S).group(1)
    return json.loads(gzip.decompress(base64.b64decode(blob)))

clips, offs, srcrank = _grab("_IQ_CLIPS_B64"), _grab("_IQ_OFFS_B64"), _grab("_IQ_SRC_B64")
hours_total = sum(c["du"] for c in clips) / 3600
resolvable = [s for s, v in offs.items() if v.get("off")]
print(f"[كتالوج اللهجة العراقية] {len(clips):,} مقطعاً | {hours_total:.0f} ساعة | "
      f"{len(resolvable):,} تسجيل مصدر قابل للجلب | {sum(offs[s]['size'] for s in resolvable)/1e9:.1f}GB لو جُلب كله")

# ---- حجم هدفك الحالي (نفس منطق الاختيار في السكربت: الجودة أولاً) ----
t = str(CFG["YODAS_HOURS"]).strip().lower()
if t in ("0", ""):
    print(f"\n[هدفك] YODAS معطّل (YODAS_HOURS=0) — الخلية 9 ستتخطاه. الأصل المتاح: {hours_total:.0f} ساعة.")
else:
    target = None if t in ("all", "كل") else float(t)
    ranked = sorted(srcrank.items(), key=lambda kv: -kv[1][2])
    chosen, h = [], 0.0
    for sid, (tar, ch, q) in ranked:
        if sid not in offs or offs[sid].get("off") is None:
            continue
        chosen.append(sid); h += ch
        if target and h >= target:
            break
    size_gb = sum(offs[s]["size"] for s in chosen) / 1e9
    label = f"{target:.0f} ساعة" if target else "الكتالوج كله"
    print(f"[هدفك] {label}: {len(chosen):,} تسجيلاً مصدراً ≈ {size_gb:.1f}GB تنزيلاً خاماً (سيُشفَّر إلى Opus للتدريب)")
print("\n[جاهز] تابع إلى خلايا البيانات (8 وما بعدها).")

<div dir="rtl">

## مرحلة البيانات — المصادر

| المصدر | ما هو | الحجم | ملاحظات |
|---|---|---|---|
| `youssef` | 3,867 مقطعاً عراقياً 22.05kHz مع نصوص | ≈ 720MB | مقاطع قصيرة نظيفة |
| `shiekh` | 827 مقطعاً 16kHz — متحدث واحد | ≈ 640MB | يساعد ثبات المتحدث الواحد |
| `safaa` | 676 ملف wav — متحدثة أنثى جاهزة لـTTS | مئات MB | نظيفة جداً |
| `lahgtna` | الجزء العراقي من Lahgtna v2: ~10,746 مقطعاً / ≈50 ساعة 16kHz مع نصوص لهجية | ≈5.8GB | جديد — يجلب عبر datasets-server بترقيم صفحات (استئناف كامل)؛ قد يتداخل جزئياً مع مصادر YODAS |
| `kharrufa` | 450 مقطعاً استوديو للهجة العراقية 44.1kHz بنصوص مشكولة كاملة | ≈2.1GB تنزيلاً | جديد — يُستخرج منه 683MB ثم يُحذف الأرشيف (Iraqi Dialect TTS Corpus، CC BY 4.0) |
| YODAS‑IQ | مقاطع عراقية مصنّفة من كتالوج YODAS3 (49,194 مقطعاً / 339 ساعة / 2,360 تسجيلاً) | حسب ساعاتك | تُجلب **بتنزيل مُجزَّأ**: نطاق البايتات المطلوب فقط من ملفات tar الضخمة — لهذا 200 ساعة = 12.7GB فقط بدل 22GB+ |
| `custom` | **ملفاتك أنت** — عبر الخلية 10 | حسبك | تسجيلات بموافقة أصحابها، نظيفة قدر الإمكان |

المرحلة 11 (التقطيع والترميز) تحوّل كل ما سبق إلى مقاطع **Opus 24kHz أحادية بطول 1–29.5 ثانية** —
الشكل الذي يقرأه مدرب Seed-VC مباشرة (يفحص المجلد بشكل تكراري: `yodas/`, `youssef/`, `lahgtna/`, `kharrufa/`, `custom/` ...).

فشل أحد المصادر الصغيرة ليس قاتلاً — الخط يتابع بالبقية ويمكن إعادة المحاولة لاحقاً (كل المصادر
تُعلَّم بعلامات `.done` عند الإنجاز فلا يُعاد تنزيلها). على Colab: إن انقطعت الجلسة أثناء التنزيل
عُيد الجلب من حيث توقف عند إعادة تشغيل الخلية — والوقت الميت الوحيد هو إعادة بناء البيئة (الخلية 5).

</div>

In [ ]:
# ============================================================ #
#  الخلية ٨ — تنزيل المجموعات العراقية النظيفة الصغيرة         #
#  (youssef / shiekh / safaa من HuggingFace — ≈2-3GB)          #
#  قابلة للاستئناف: أعد التشغيل لتكملة ما فشل فقط              #
# ============================================================ #
import sys
if CFG["SKIP_SMALL"]:
    print("تخطي المجموعات النظيفة الخمس (SKIP_SMALL=True) — إن أردت فعّلها من الخلية 2.")
else:
    import shutil
    free0 = shutil.disk_usage(ROOT).free / 1e9
    print(f"المساحة الحرة قبل التنزيل: {free0:.1f}GB "
          "(المرحلة كلها ≈7.5GB، وأكبر عنصر فيها lahgtna)")
    sh([sys.executable, PIPE, "small",
        "--root", ROOT,
        "--workers", int(CFG["FETCH_WORKERS"])])
    AUD = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".webm"}
    for name, note in (("youssef", "مقاطع قصيرة نظيفة"),
                       ("shiekh", "متحدث ديني واحد"),
                       ("safaa", "صوت أنثى عراقي"),
                       ("lahgtna", "≈50 ساعة عراقية إضافية — جديد"),
                       ("kharrufa", "450 مقطع استوديو — جديد")):
        d = ROOT / "raw" / name
        n = len([p for p in d.iterdir() if p.suffix.lower() in AUD]) if d.exists() else 0
        print(f"  {name:<8}: {n:,} ملفاً | {note}")
    free1 = shutil.disk_usage(ROOT).free / 1e9
    print(f"استُهلك {free0-free1:.1f}GB | الحر الآن {free1:.1f}GB")
    st = ROOT / "raw" / "_lahgtna_state.json"
    if st.exists():
        import json
        s = json.loads(st.read_text())
        fp, fa = len(s.get("failed_pages", [])), len(s.get("failed_audio", []))
        if fp or fa:
            print(f"[ملاحظة] lahgtna: {fp} صفحة و{fa} مقطعاً فشلا مؤقتاً — "
                  "أعد هذه الخلية لإعادة المحاولة فقط (الموجود لا يُعاد)")
print("\n[جاهز] تابع إلى الخلية 9 (YODAS) — أو الخلية 10 لدمج ملفاتك الخاصة.")

In [ ]:
# ============================================================ #
#  الخلية ٩ — جلب بيانات YODAS العراقية (الكتلة الكبرى)        #
#  تنزيل مُجزَّأ متوازٍ من أرشيفات espnet/yodas3                #
#  يستغرق عادة 15-60 دقيقة (حسب الشبكة وساعاتك)                #
#  قابلاً للاستئناف 100%: أعد التشغيل ليكمل من حيث توقف         #
# ============================================================ #
import sys
import shutil

t = str(CFG["YODAS_HOURS"]).strip().lower()
if t == "0":
    print("YODAS معطّل (YODAS_HOURS=0) — تخطّي.")
else:
    free0 = shutil.disk_usage(ROOT).free / 1e9
    print(f"المساحة الحرة قبل الجلب: {free0:.1f}GB")
    sh([sys.executable, PIPE, "yodas",
        "--root", ROOT,
        "--hours", str(CFG["YODAS_HOURS"]),
        "--workers", int(CFG["FETCH_WORKERS"])])
    n = len(list((ROOT / "raw" / "yodas_webm").glob("*.webm"))) if (ROOT / "raw" / "yodas_webm").exists() else 0
    free1 = shutil.disk_usage(ROOT).free / 1e9
    print(f"\nتسجيلات مصدر جاهزة: {n:,} ملف webm | استُهلك {free0-free1:.1f}GB | الحر الآن {free1:.1f}GB")
    state = ROOT / "raw" / "yodas_fetch_state.json"
    if state.exists():
        import json
        failed = json.loads(state.read_text()).get("failed", [])
        if failed:
            print(f"مصادر فشل جلبها: {len(failed)} — أعد هذه الخلية لإعادة محاولة المتبقي، أو تجاهل (بقية البيانات تكفي)")
print("\n[جاهز] تابع إلى الخلية 10 (اختيارية) ثم 11 (التقطيع والترميز).")

In [ ]:
# ============================================================ #
#  الخلية ١٠ — (اختيارية) دمج تسجيلاتك الخاصة في التدريب        #
#  على Colab: الشريط الجانبي الأيسر <- أيقونة المجلد <- تنقّل   #
#  إلى /content/iraqi_seedvc/raw/custom <- زر الرفع (أعلى       #
#  اللوحة) ثم أعد تشغيل هذه الخلية                             #
#  ستُقطَّع تلقائياً إلى مقاطع 1-29.5 ثانية وتُرمَّز Opus 24kHz  #
# ============================================================ #
import sys, os, json, time
from multiprocessing import Pool
from pathlib import Path

CUSTOM = ROOT / "raw" / "custom"
CUSTOM.mkdir(parents=True, exist_ok=True)
EXTS = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".webm"}
files = [p for p in sorted(CUSTOM.iterdir()) if p.is_file() and p.suffix.lower() in EXTS]
print(f"مجلد بياناتك: {CUSTOM}")
print(f"ملفات صوتية موجودة: {len(files)}")

if not files:
    print('''
لا توجد ملفات بعد — لتفعيل ذلك:
  1) من لوحة الملفات الجانبية في Colab: تنقّل إلى المجلد أعلاه ثم ارفع ملفاتك بزر الرفع
     (يدعم: wav / flac / mp3 / m4a / opus / ogg — الرفع الكبير يستغرق دقائق)
  2) الأفضل: كلام نظيف بلا موسيقى أو ضجيج، 1-30 ثانية أو أطول (تُقطَّع تلقائياً)
  3) أعد تشغيل هذه الخلية، ثم شغّل خلية التقطيع (11) لتشمل بياناتك
(تخطَّ هذه الخلية إن لم يكن لديك بيانات خاصة — مجموعة YODAS + الصغيرة كافية تماماً)''')
else:
    sys.path.insert(0, str(ROOT))
    import iraqi_pipeline as P   # خط البيانات المستخرج في الخلية 7 — نفس منطق التقطيع المجرَّب
    import types
    a = types.SimpleNamespace(out=str(DATA_DIR))
    jobs = []
    for f in files:
        jobs.extend(P._prep_job_src(a, str(f), "custom"))
    print(f"سيُنتج عنها {len(jobs)} مقطعاً (يُتخطى الموجود منها)")
    t0, ok_n, bad_n = time.time(), 0, 0
    with Pool(processes=int(CFG["PREP_WORKERS"] or 0) or min(16, os.cpu_count() or 4)) as pool:
        for out, dur, okk in pool.imap_unordered(P._run_prep_job, jobs, chunksize=16):
            if okk:
                ok_n += 1
            else:
                bad_n += 1
    print(f"تم: {ok_n} مقطعاً ناجحاً | {bad_n} فاشلاً في {time.time()-t0:.0f} ثانية")
    # تحديث فهرس بيانات التدريب (كما يفعل أمر prep نفسه)
    idx = DATA_DIR / "_prep_index.jsonl"
    prev = {}
    if idx.exists():
        for line in idx.read_text().splitlines():
            try:
                r = json.loads(line)
                prev[r["file"]] = r["dur"]
            except Exception:
                pass
    for j in jobs:
        prev[os.path.relpath(j[3], DATA_DIR)] = j[2]
    prev = {f: d for f, d in prev.items() if (DATA_DIR / f).exists()}
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    idx.write_text("".join(json.dumps({"file": f, "dur": d}) + "\n" for f, d in sorted(prev.items())))
    print("فهرس بيانات التدريب محدّث — أكمل إلى الخلية 11.")

In [ ]:
# ============================================================ #
#  الخلية ١١ — التقطيع والترميز: بناء بيانات التدريب النهائية  #
#  كل المقاطع -> Opus 24kHz أحادي، 1-29.5 ثانية                #
#  (يشمل YODAS + المجموعات الخمس + بياناتك إن وضعتها)          #
#  قابلة للاستئناف: الموجود يُتخطى                              #
# ============================================================ #
import sys
import os

workers = int(CFG["PREP_WORKERS"] or 0) or min(16, os.cpu_count() or 4)
sh([sys.executable, PIPE, "prep",
    "--root", ROOT,
    "--out", DATA_DIR,
    "--workers", workers])

n = sum(1 for _ in DATA_DIR.rglob("*.ogg"))
print(f"\nمقاطع التدريب الجاهزة: {n:,} ملف Opus في {DATA_DIR}")
assert n >= 50, f"مقاطع قليلة جداً ({n}) — تأكد أن الخليتين 8 و9 أنزلتا بيانات ثم أعد المحاولة"
print("[جاهز] البيانات مبنية — تابع إلى الخلية 12 (تنزيل نقاط النموذج).")

In [ ]:
# ============================================================ #
#  الخلية ١٢ — تنزيل نقاط النموذج الأساسية مسبقاً (≈3.5GB)     #
#  DiT base f0 44k + Whisper-base + BigVGAN-44k + RMVPE +      #
#  CAMPPlus + OpenVoice + se_db — كلها في checkpoints/seed-vc  #
#  (يمنع تعطل بدء التدريب لاحقاً بسبب التنزيل، وقابل للاستئناف)#
# ============================================================ #
import sys
sh([sys.executable, PIPE, "prefetch", "--repo", SEEDVC])
import shutil
ck = SEEDVC / "checkpoints"
total = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e9 if ck.exists() else 0
print(f"\nحجم مخزون النقاط: {total:.2f}GB في {ck}")
print("[جاهز] كل شيء مطلوب للتدريب موجود — تابع إلى مرحلة التدريب.")

<div dir="rtl">

## مرحلة التدريب — كيف تعمل؟ (بمنطق Colab)

- **نقطة البداية دائماً** الأوزان المنشورة `DiT_seed_v2_uvit_whisper_base_f0_44k_bigvgan_pruned_ft_ema.pth`
  من `Plachta/Seed-VC` — **نموذج base مع شرط النبرة F0 وترميز 44.1kHz** (نحو 78M باراميتر): يحافظ على
  النبرة العراقية بدقة أعلى بكثير من نموذج tiny. لا تدريب من الصفر أبداً؛ وإن وُجدت نقطة سابقة في مجلد
  التشغيل يُستأنف منها تلقائياً (يُحتفظ بآخر نقطتين `DiT_epoch_*_step_*.pth` — سلوك المدرب الرسمي).
- **السجل**: سطر `epoch E, step S, loss: L` كل 10 خطوات؛ نقطة حفظ كل `SAVE_EVERY` خطوة (500 افتراضياً)؛
  وفي النهاية يُكتب `ft_model.pth` (أوزان DiT المضبوطة — وهي كل ما يلزم للنشر والاستدلال).
- **الإعدادات الموصى بها من دليل المستودع (FINETUNING.md)**: دفعة 1 لذاكرة 16GB (T4) — النموذج أكبر
  والدفعة الصغيرة أمتن مع صدمات الدفعة؛ **3000–6000 خطوة لتكيّف عام مع اللهجة** (3000 افتراضياً هنا)؛
  معدل التعلم مثبّت في الإعداد الرسمي (1e-5 مع ExponentialLR) — لا ترفعه للضبط الدقيق.
- **لماذا التدريب داخل النواة؟** الخلية 14 تستورد وحدة `train.py` الرسمية وتشغلها في نواة الدفتر نفسها
  (بلا أي عملية ابن) — فالتقدم الحي (شريط tqdm والخسارة) يظهر في الخلية مباشرة، والسجل نفسه يُكتب في
  `train_console.log` للمراقبة بالخليتين 15/16. لو قُطعت الجلسة، تُستأنف النقاط من Drive بالخلية 13.
- **مرآة Drive الحية**: أثناء التدريب يعمل خيط خفيف يزامن كل نقطة جديدة إلى
  `MyDrive/rafidain_seedvc/runs/<اسم-التشغيل>/` خلال دقيقة من حفظها — أي أن أقصى خسارة عند أي انقطاع
  جلسة = آخر `SAVE_EVERY` خطوة فقط (500 خطوة ≈ دقائق معدودة على T4).
- **انقطعت الجلسة؟** افتح الدفتر ← **Run all**: تُثبَّت الاعتماديات داخل النواة (دقائق عبر uv)، وتستعيد
  الخلية 13 النقاط من Drive، ويستأنف التدريب من آخر نقطة حفظ. البيانات الخام إن فُقد القرص المحلي
  تُجلب من جديد (استئناف كامل).

مدة تقديرية: 3000 خطوة على **T4 في حدود 4–7 ساعات تقريباً** (النموذج الأكبر أدق لكنه أبطأ)، وعلى
**L4 نحو نصف ذلك**. راقب معدل الخطوات في خلية المراقبة بعد أول دقيقة لتقدير زمنك الفعلي بدقة.

</div>

In [ ]:
# ============================================================ #
#  الخلية ١٣ — استعادة نقاط التدريب من Drive (بعد أي انقطاع)   #
#  تعيد أحدث نقاط التفتيش + السجلات من نسخة Drive الاحتياطية   #
#  إلى القرص المحلي فيستأنف التدريب من حيث توقف                #
#  (خلية استباقية: لا تفعل شيئاً إن كان المحلي أحدث أو لا نسخة)#
# ============================================================ #
import shutil, re
from pathlib import Path

DRIVE_OK   = globals().get("DRIVE_OK", False)
DRIVE_ROOT = globals().get("DRIVE_ROOT", None)

def _step_of(p: Path):
    m = re.search(r"step_(\d+)", p.stem)
    return int(m.group(1)) if m else -1

loc_ckpts = sorted(RUN_DIR.glob("DiT_epoch_*_step_*.pth"), key=_step_of)
loc_ft    = RUN_DIR / "ft_model.pth"
drv       = (DRIVE_ROOT / "runs" / CFG["RUN_NAME"]) if (DRIVE_OK and DRIVE_ROOT) else None

if loc_ckpts or loc_ft.exists():
    last = f" (حتى الخطوة {_step_of(loc_ckpts[-1])})" if loc_ckpts else ""
    print(f"[محلي] توجد نقاط محلية أصلاً{last} — لا حاجة للاستعادة.")
    print("        (لفرض الاستعادة من Drive احذف مجلد التشغيل المحلي ثم أعد هذه الخلية)")
elif drv and drv.exists():
    files = [p for p in drv.iterdir() if p.is_file()] if drv.is_dir() else []
    if not files:
        print(f"[Drive] لا توجد نسخة احتياطية سابقة في {drv} — تشغيل جديد كلياً.")
    else:
        print(f"[Drive] استعادة {len(files)} ملفاً من {drv} ...")
        RUN_DIR.mkdir(parents=True, exist_ok=True)
        for f in files:
            print(f"  + {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
            shutil.copy2(f, RUN_DIR / f.name)
        ck = sorted(RUN_DIR.glob("DiT_epoch_*_step_*.pth"), key=_step_of)
        if ck:
            print(f"\n[تم] أُعيدت أحدث نقطة إلى الخطوة {_step_of(ck[-1])} — التدريب سيكمل منها (الخلية 14).")
        elif (RUN_DIR / "ft_model.pth").exists():
            print("\n[تم] النموذج النهائي ft_model.pth مستعاد — يمكنك القفز إلى التحقق (الخلية 17).")
        else:
            print("\n[تم] استُعيدت السجلات فقط — سيبدأ التدريب من الصفر.")
else:
    print("[استعادة] لا نسخة Drive (غير موصول أو لا سجل سابق) — تشغيل جديد كلياً.")

In [ ]:
# ============================================================ #
#  الخلية ١٤ — إطلاق التدريب داخل نواة الدفتر مباشرة           #
#  (استيراد وحدة train.py الرسمية — بلا أي عملية ابن مستقلة)   #
#  التقدم يظهر حياً في الخلية + يُسجل في train_console.log     #
#  ومرآة Drive تنسخ كل نقطة فور حفظها أثناء التدريب           #
# ============================================================ #
import subprocess, os, time, sys, threading
from pathlib import Path

LOG  = RUN_DIR / "train_console.log"
FT   = RUN_DIR / "ft_model.pth"
CFG_MODEL = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml"

# ---- 0) كتابة مزامن النسخ إلى Drive (stdlib فقط — يعمل بأي بايثون) ----
SYNC_SCRIPT = ROOT / "drive_sync_loop.py"
SYNC_SCRIPT.write_text(r'''
import os, sys, time, shutil
from pathlib import Path

RUN_DIR  = Path(sys.argv[1])   # مجلد التشغيل المحلي
DRV_DIR  = Path(sys.argv[2])   # مجلد النسخ على Drive
TRAIN_PID = int(sys.argv[3])   # PID عملية التدريب (0 = بلا فحص حياة)
INTERVAL = int(os.environ.get("SYNC_INTERVAL", "300"))
KEEP     = int(os.environ.get("SYNC_KEEP", "2"))
ONCE     = "--once" in sys.argv

def _step(p):
    try:    return int(p.stem.rsplit("_", 1)[-1])
    except Exception: return -1

def _alive(pid):
    if not pid:
        return False
    try:
        os.kill(pid, 0); return True
    except Exception:
        return False

def sync_once():
    if not RUN_DIR.exists():
        return 0
    DRV_DIR.mkdir(parents=True, exist_ok=True)
    n = 0
    # نقاط DiT: الأحدث KEEP فقط (كل نقطة تحمل net+optimizer+scheduler)
    ck = sorted(RUN_DIR.glob("DiT_epoch_*_step_*.pth"), key=_step)
    for p in ck[-KEEP:]:
        d = DRV_DIR / p.name
        if (not d.exists()) or (d.stat().st_mtime < p.stat().st_mtime):
            shutil.copy2(p, d); n += 1
    for p in sorted(DRV_DIR.glob("DiT_epoch_*_step_*.pth"), key=_step)[:-KEEP]:
        try: p.unlink()
        except Exception: pass
    # الملفات الصغيرة: دائماً
    for name in ("ft_model.pth", "train_console.log", "loss_curve.png",
                 "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml"):
        s = RUN_DIR / name
        if s.exists():
            d = DRV_DIR / name
            if (not d.exists()) or (d.stat().st_mtime < s.stat().st_mtime):
                shutil.copy2(s, d); n += 1
    return n

if ONCE:
    print(f"[sync] {sync_once()} file(s) updated on Drive")
    sys.exit(0)

print(f"[sync] drive backup watcher: every {INTERVAL}s, keep last {KEEP} ckpts -> {DRV_DIR}", flush=True)
while True:
    try:
        n = sync_once()
        if n:
            print(f"[sync] {time.strftime('%H:%M:%S')} {n} file(s) -> Drive", flush=True)
    except Exception as e:
        print(f"[sync] skip cycle ({e})", flush=True)
    if not _alive(TRAIN_PID):
        try: sync_once()
        except Exception: pass
        print("[sync] training process ended - final sync done, watcher exits", flush=True)
        break
    time.sleep(INTERVAL)
''', encoding="utf-8")

BATCH = int(CFG.get("_BATCH") or 1)
MAXS  = int(CFG["STEPS"])

# ---- 1) إطلاق التدريب داخل النواة مباشرة (import train — بلا أي عملية ابن) ----
# لماذا داخل النواة؟ التدريب كعملية ابن مستقلة يفقد سياق النواة عند إعادة تشغيلها
# ويعقّد الاستئناف؛ أما بالاستيراد المباشر فالتقدم يظهر حياً في الخلية نفسها،
# والنقاط تُحفظ على القرص فتُستعاد بأمان من أي جلسة جديدة (الخلية 13).
if FT.exists():
    print(f"النموذج النهائي موجود أصلاً: {FT}")
    print("لتدريب جديد كلياً: غيّر RUN_NAME في الخلية 2 (أو احذف ft_model.pth ثم أعد هذه الخلية).")
else:
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # ---- مرآة Drive الحية: خيط خفيف يزامن كل نقطة أثناء التدريب داخل النواة ----
    _DRV = (DRIVE_ROOT / "runs" / CFG["RUN_NAME"]) if (globals().get("DRIVE_OK") and globals().get("DRIVE_ROOT") and CFG.get("DRIVE_BACKUP", True)) else None
    if _DRV:
        _keep = str(int(CFG.get("DRIVE_CKPT_KEEP", 2)))
        def _drive_mirror(stop_evt, drv=_DRV, keep=_keep):
            # يستدعي drive_sync_loop.py --once كل 60 ثانية حتى نهاية التدريب
            import subprocess as _sp
            env_s = dict(ENV); env_s["SYNC_KEEP"] = keep
            while not stop_evt.wait(60):
                try:
                    _sp.run([sys.executable, str(SYNC_SCRIPT), str(RUN_DIR), str(drv), "0", "--once"],
                            capture_output=True, text=True, timeout=300, env=env_s)
                except Exception:
                    pass
        _stop_evt = threading.Event()
        _mirror = threading.Thread(target=_drive_mirror, args=(_stop_evt,), daemon=True)
        _mirror.start()
        print(f"[train] مرآة Drive مفعّلة: كل نقطة تُنسخ خلال دقيقة إلى {_DRV} (أبقِ آخر {int(CFG.get('DRIVE_CKPT_KEEP', 2))})")
    else:
        _stop_evt = None
        print("[train] Drive غير موصول — النقاط تبقى محلية فقط.")
    print(f"[train] بدء الضبط الدقيق: {MAXS} خطوة، دفعة {BATCH}، حفظ كل {int(CFG['SAVE_EVERY'])}")
    print(f"[train] التشغيل داخل نواة الدفتر مباشرة (import train.Trainer) — بلا أي عملية ابن")
    # train.py يعمل من مجلد المستودع (مساراته نسبية: configs/, checkpoints/, runs/)
    import os as _os, sys as _sys
    _os.chdir(SEEDVC)
    _sys.path.insert(0, str(SEEDVC))
    import train as SEED_TRAIN   # وحدة المدرب الرسمية — نفسها التي تستخدمها train.py

    trainer = SEED_TRAIN.Trainer(
        config_path=str(CFG_MODEL),
        pretrained_ckpt_path=None,     # None = يكتشف أحدث نقطة في مجلد التشغيل تلقائياً (استئناف)
        data_dir=str(DATA_DIR),
        run_name=CFG["RUN_NAME"],
        batch_size=BATCH,
        steps=MAXS,
        max_epochs=1000,               # السقف الفعلي هو MAXS (يتوقف عنده)
        save_interval=int(CFG["SAVE_EVERY"]),
        num_workers=int(CFG["TRAIN_WORKERS"]),
        device="cuda:0",
    )
    # توجيه مخرجات السجلات إلى ملف السجل أيضاً (نفس ملف المراقبة في الخلية 15)
    # ملاحظة: نحفظ مرجع sys.stdout الأصلي أولاً — print تكتب فيه مباشرة فيمنع أي تكرار
    import functools
    _logfh = open(LOG, "a", buffering=1)
    _old_out, _old_err = _sys.stdout, _sys.stderr
    _orig_print = functools.partial(print, file=_old_out)
    class _Tee:
        def write(self, s):
            _logfh.write(s)
            _old_out.write(s)
        def flush(self):
            _logfh.flush(); _old_out.flush()
        def isatty(self):
            return False
    _sys.stdout = _Tee(); _sys.stderr = _Tee()
    try:
        trainer.train()
    finally:
        _sys.stdout, _sys.stderr = _old_out, _old_err
        _logfh.close()
        if _stop_evt is not None:
            _stop_evt.set()   # أوقف مرآة Drive
    print(f"\n[train] اكتمل — النموذج النهائي: {RUN_DIR / 'ft_model.pth'}")

# ---- 2) مزامنة نهائية فورية إلى Drive (التدريب انتهى داخل النواة — لا حاجة لمراقب خلفي) ----
DRV_RUN = (DRIVE_ROOT / "runs" / CFG["RUN_NAME"]) if (globals().get("DRIVE_OK") and globals().get("DRIVE_ROOT")) else None
if DRV_RUN and SYNC_SCRIPT.exists():
    sh([sys.executable, str(SYNC_SCRIPT), str(RUN_DIR), str(DRV_RUN), "0", "--once"], check=False)
    print(f"[نسخ احتياطي] نُسخت النقاط والسجلات إلى Drive: {DRV_RUN}")
elif not DRV_RUN:
    print("[نسخ احتياطي] Drive غير موصول — النقاط تبقى محلية فقط.")

if FT.exists():
    print("\nالخطوة التالية: خلية «التحقق» (17) للاستماع إلى النموذج، ثم التصدير (18).")

In [ ]:
# ============================================================ #
#  الخلية ١٥ — مراقبة التدريب (أعد تشغيلها متى شئت)            #
#  تقدم + معدل + زمن متبقٍ + منحنى الخسارة + آخر السجل          #
#  + حالة النسخ الاحتياطي إلى Drive                            #
#  (تسميات الرسم بالإنجليزية لأن matplotlib لا يشكّل            #
#   الحروف العربية المتصلة بشكل صحيح)                          #
# ============================================================ #
import re, os, time, subprocess, sys
from pathlib import Path
from IPython.display import display, Image as IPyImage

LOG  = RUN_DIR / "train_console.log"
FT   = RUN_DIR / "ft_model.pth"
STEP_RE = re.compile(r"epoch (\d+), step (\d+), loss: ([0-9.eE+-]+)")
MAXS = int(CFG["STEPS"])

def _running_in_kernel():
    # التدريب داخل النواة: كائن المدرب حي في ذاكرة الجلسة
    t = globals().get("trainer")
    return t is not None and getattr(t, "iters", None) is not None

lines = LOG.read_text(errors="replace").splitlines() if LOG.exists() else []
pts = []
for ln in lines:
    m = STEP_RE.search(ln)
    if m:
        pts.append((int(m.group(2)), float(m.group(3))))

if FT.exists():
    status = "منجز — ft_model.pth موجود (انتقل إلى خلية التحقق 17)"
elif _running_in_kernel():
    status = "يعمل الآن (داخل النواة)"
elif pts:
    status = "متوقف قبل الاكتمال (راجع آخر السجل — أعد الخلية 14 للاستئناف من آخر نقطة)"
else:
    status = "لم يبدأ بعد"

steps = pts[-1][0] if pts else 0
start = LOG.stat().st_mtime if LOG.exists() else time.time()
elapsed = max(time.time() - start, 1.0)
rate = steps / elapsed if steps else 0.0
eta_min = (MAXS - steps) / rate / 60 if rate > 0 else float("nan")
pct = min(100.0 * steps / MAXS, 100.0) if MAXS else 0.0
filled = int(30 * pct / 100)
bar = "#" * filled + "-" * (30 - filled)

print(f"الحالة: {status}")
print(f"[{bar}] {pct:5.1f}%   step {steps}/{MAXS}   معدل {rate:.2f} خطوة/ث   زمن منقضٍ {elapsed/60:.0f} د   متبقٍ ≈ {eta_min:.0f} د")
if pts:
    print(f"آخر خسارة (EMA): {pts[-1][1]:.4f} عند الخطوة {steps}")

# ---- منحنى الخسارة ----
if len(pts) >= 2:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    xs, ys = zip(*pts)
    fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
    ax.plot(xs, ys, lw=1.3, color="#d62728")
    ax.set_xlabel("step")
    ax.set_ylabel("EMA loss")
    ax.set_title(f"loss curve - {CFG['RUN_NAME']} ({len(pts)} points)")
    ax.grid(alpha=0.3)
    chart = RUN_DIR / "loss_curve.png"
    fig.savefig(chart, dpi=110)
    plt.close(fig)
    display(IPyImage(filename=str(chart)))
    print(f"(حُفظ أيضاً بشكل دائم: {chart})")

# ---- آخر السجل ----
print("\n— آخر 12 سطراً من train_console.log —")
for ln in lines[-12:]:
    print("  " + ln)
ckpts = sorted(RUN_DIR.glob("DiT_epoch_*_step_*.pth"), key=lambda p: int(p.stem.rsplit("_", 1)[-1]))
if ckpts:
    print(f"\nنقاط التفتيش المحفوظة (آخر 3): " + ", ".join(c.name for c in ckpts[-3:]))

# ---- حالة النسخ الاحتياطي إلى Drive ----
SYNC_SCRIPT = ROOT / "drive_sync_loop.py"
DRV_RUN = (globals().get("DRIVE_ROOT") / "runs" / CFG["RUN_NAME"]) if (globals().get("DRIVE_OK") and globals().get("DRIVE_ROOT")) else None
if DRV_RUN and SYNC_SCRIPT.exists():
    try:
        r = subprocess.run([sys.executable, str(SYNC_SCRIPT), str(RUN_DIR), str(DRV_RUN), "0", "--once"],
                           capture_output=True, text=True, timeout=900)
        drv_ck = sorted(DRV_RUN.glob("DiT_epoch_*_step_*.pth"), key=lambda p: int(p.stem.rsplit("_", 1)[-1])) if DRV_RUN.exists() else []
        print(f"\n[Drive] النسخة الاحتياطية: {len(drv_ck)} نقطة على Drive"
              + (f" (أحدثها حتى الخطوة {int(drv_ck[-1].stem.rsplit('_', 1)[-1])})" if drv_ck else " — تُنشئ عند أول حفظ نقطة")
              + f" | {r.stdout.strip()}")
    except Exception as e:
        print(f"\n[Drive] تعذرت المزامنة اللحظية ({e}) — المراقب الخلفي (الخلية 14) يكفي عادة.")

In [ ]:
# ============================================================ #
#  الخلية ١٦ — (اختيارية) مراقبة تقدم التدريب أثناء عمله        #
#  تُشغَّل في خلية ثانية أثناء دوران التدريب في الخلية 14:      #
#  تحديث حي كل 30 ثانية + مزامنة Drive كل دورة                 #
#  ملاحظة: النشاط هنا لا يبقي الجلسة حية بحد ذاته — الخلية 14  #
#  النشطة (التدريب داخل النواة) هي التي تمنع القطع الخامل      #
# ============================================================ #
import re, os, time, subprocess, sys
from pathlib import Path
from IPython.display import clear_output

LOG  = RUN_DIR / "train_console.log"
FT   = RUN_DIR / "ft_model.pth"
STEP_RE = re.compile(r"epoch (\d+), step (\d+), loss: ([0-9.eE+-]+)")
MAXS = int(CFG["STEPS"])
SYNC_SCRIPT = ROOT / "drive_sync_loop.py"
DRV_RUN = (globals().get("DRIVE_ROOT") / "runs" / CFG["RUN_NAME"]) if (globals().get("DRIVE_OK") and globals().get("DRIVE_ROOT")) else None

def _running_in_kernel():
    t = globals().get("trainer")
    return t is not None and getattr(t, "iters", None) is not None

def _drive_sync():
    if DRV_RUN and SYNC_SCRIPT.exists():
        try:
            subprocess.run([sys.executable, str(SYNC_SCRIPT), str(RUN_DIR), str(DRV_RUN), "0", "--once"],
                           capture_output=True, text=True, timeout=600)
        except Exception:
            pass

deadline = time.time() + 12 * 3600   # مهلة قصوى 12 ساعة (سقف جلسة Colab)
while time.time() < deadline:
    clear_output(wait=True)
    lines = LOG.read_text(errors="replace").splitlines() if LOG.exists() else []
    steps, loss = 0, None
    for ln in lines:
        m = STEP_RE.search(ln)
        if m:
            steps, loss = int(m.group(2)), float(m.group(3))
    pct = min(100.0 * steps / MAXS, 100.0)
    bar = "#" * int(30 * pct / 100)
    msg = f"[{bar:<30}] {pct:5.1f}%   step {steps}/{MAXS}"
    if loss is not None:
        msg += f"   loss {loss:.4f}"
    print(msg, flush=True)
    _drive_sync()
    if FT.exists():
        print("\nاكتمل التدريب — ft_model.pth جاهز. انتقل إلى خلية التحقق (17).")
        _drive_sync()
        break
    if not _running_in_kernel():
        print("\nالتدريب متوقف قبل الاكتمال (الخلية 14 انتهت أو قُطعت).")
        print("شغّل خلية المراقبة (15) لرؤية آخر السجل — أعد الخلية 14 للاستئناف من آخر نقطة.")
        break
    time.sleep(30)
else:
    print("انتهت مهلة المراقبة (12 ساعة) — استخدم الخلية 15 للفحص اليدوي.")
    print("إن كانت الجلسة ستُقطع قريباً: نقاطك محفوظة على Drive وتُستعاد بالخلية 13.")

<div dir="rtl">

## التحقق والتصدير

قبل الاحتفال بالنموذج نجري **استدلالاً حقيقياً** بالنقطة الناتجة: مقطع مصدر من أمثلة Seed-VC يُحوَّل
بصوت مرجعي عراقي من بياناتك المجهزة، وتستمع للناتج داخل الدفتر مباشرة. بعدها نبني **حزمة التصدير**
وننسخها فوراً إلى Drive:

```
exports/iraqi_v1/                          (محلي + MyDrive/rafidain_seedvc/exports/iraqi_v1/)
├── seed_uvit_whisper_base_f0_44k_iraqi.pth      الأوزان المضبوطة (للنشر المباشر)
├── config_dit_mel_seed_uvit_whisper_base_f0_44k.yml
├── training_report.json                   تقرير التدريب (البيانات والكتالوج والتاريخ)
├── _stats.json                            ساعات/ملفات كل مصدر
├── train_console.log                      سجل التدريب كاملاً
├── loss_curve.png                         منحنى الخسارة
└── README.md                              أوامر الاستخدام الجاهزة
```

</div>

In [ ]:
# ============================================================ #
#  الخلية ١٧ — التحقق: استدلال حقيقي بالنقطة المضبوطة          #
#  (مصدر من أمثلة Seed-VC + مرجع عراقي من بياناتك -> تشغيل)    #
# ============================================================ #
import sys
from IPython.display import Audio, display
from pathlib import Path

FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — شغّل خلية الإطلاق (14) وراقب حتى الاكتمال"
CFG_MODEL = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml"

sh([sys.executable, PIPE, "verify",
    "--repo", SEEDVC,
    "--run-dir", RUN_DIR,
    "--ckpt", FT,
    "--config", CFG_MODEL,
    "--data-dir", DATA_DIR], check=False)   # فشل verify ليس قاتلاً — الأوزان تصدَّر على أي حال

outs = sorted((RUN_DIR / "verify_out").glob("vc_*.wav"))
if outs:
    print(f"\nناتج التحويل: {outs[0].name}  ({outs[0].stat().st_size/1e6:.1f} MB)")
    print("استمع للناتج (المصدر إنجليزي يحمل الطابع الصوتي العراقي المرجعي):")
    display(Audio(filename=str(outs[0])))
    for extra in outs[1:3]:
        print(f"ناتج إضافي: {extra.name}")
        display(Audio(filename=str(extra)))
else:
    print("لم يُنتج ملف صوتي — راجع السجل أعلاه. (غير قاتل: الأوزان ستصدَّر في الخلية 18 على أي حال)")

In [ ]:
# ============================================================ #
#  الخلية ١٨ — تصدير الحزمة النهائية                           #
#  أوزان + إعداد + تقرير + سجلات — محلياً ثم نسخها إلى Drive   #
# ============================================================ #
import sys
import shutil
from pathlib import Path

FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — أكمل التدريب أولاً"
EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
CFG_MODEL = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml"

sh([sys.executable, PIPE, "export",
    "--run-dir", RUN_DIR,
    "--config", CFG_MODEL,
    "--data-dir", DATA_DIR,
    "--out", EXPORT_DIR])

# توثيق إضافي يُنسخ مع الحزمة (قابلية إعادة الإنتاج جزء من الجودة)
for extra in (RUN_DIR / "train_console.log", RUN_DIR / "loss_curve.png",
              RUN_DIR / "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml", DATA_DIR / "_stats.json"):
    if extra.exists():
        shutil.copy2(extra, EXPORT_DIR / extra.name)
        print(f"  + {extra.name}")

total = 0
print(f"\nحزمة التصدير (محلي): {EXPORT_DIR}")
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        s = p.stat().st_size
        total += s
        print(f"  {p.name:<46} {s/1e6:9.2f} MB")
print(f"  {'الإجمالي':<46} {total/1e6:9.2f} MB")

# ---- النسخ الفوري إلى Drive (ديمومة فورية) ----
DRV_EXP = (globals().get("DRIVE_ROOT") / "exports" / CFG["RUN_NAME"]) if (globals().get("DRIVE_OK") and globals().get("DRIVE_ROOT")) else None
if DRV_EXP:
    print(f"\nنسخ الحزمة إلى Drive: {DRV_EXP} ...")
    DRV_EXP.mkdir(parents=True, exist_ok=True)
    for p in sorted(EXPORT_DIR.rglob("*")):
        if p.is_file():
            shutil.copy2(p, DRV_EXP / p.name)
    print("[Drive] الحزمة محفوظة بشكل دائم — تنجو من أي انقطاع جلسة.")

In [ ]:
# ============================================================ #
#  الخلية ١٩ — أرشيف tar.gz واحد (في Drive + محلي)             #
# ============================================================ #
import shutil
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
assert (EXPORT_DIR / "seed_uvit_whisper_base_f0_44k_iraqi.pth").exists(), "شغّل خلية التصدير (18) أولاً"

base = EXPORTS / f"rafidain_seedvc_{CFG['RUN_NAME']}"
archive = shutil.make_archive(str(base), "gztar", root_dir=str(EXPORTS), base_dir=EXPORT_DIR.name)
size_mb = Path(archive).stat().st_size / 1e6
print(f"الأرشيف جاهز: {archive}  ({size_mb:.1f} MB)")

# ---- نسخه إلى Drive ----
if globals().get("DRIVE_OK") and globals().get("DRIVE_ROOT"):
    drv_arc = globals()["DRIVE_ROOT"] / "exports" / Path(archive).name
    shutil.copy2(archive, drv_arc)
    print(f"[Drive] نسخة دائمة: {drv_arc}")

print('''
لتنزيله إلى جهازك:
  • من Drive (الأسهل): drive.google.com <- مجلد rafidain_seedvc/exports <- نزّل الملف
  • أو من لوحة ملفات Colab (الشريط الجانبي): مجلد exports <- كليك يمين <- Download
  • أو أضف في خلية جديدة:  from google.colab import files; files.download("<مسار الأرشيف>")
''')

<div dir="rtl">

## الحفظ الدائم — ثلاث طبقات (حتى تنزّلها بعد سنة)

| الطبقة | أين تبقى | كم تدوم | كيف تنزّلها لاحقاً |
|---|---|---|---|
| 1. **Google Drive** | `MyDrive/rafidain_seedvc/` — نسخ احتياطية أثناء التدريب + حزمة التصدير + الأرشيف | دائمة في حسابك مهما انقطعت جلسات Colab | من أي متصفح/جهاز عبر drive.google.com |
| 2. **HuggingFace Hub** | مستودع في حسابك (خاص افتراضياً) | **دائمة مجاناً — بلا علاقة بـ Google إطلاقاً** | `hf download <repo>` أو زر التنزيل من صفحة المستودع |
| 3. القرص المحلي `/content` | البيانات الخام + بيانات التدريب + الاعتماديات | تضيع عند انقطاع/إعادة تعيين الجلسة | (قابلة لإعادة البناء بالخلايا 5-11 — كلها استئنافية) |

> **الوصية**: الطبقة 2 هي ضمانتك الحقيقية للمدى البعيد — حتى لو فقدت حساب Google أو حُذف Drive،
> يبقى النموذج في HuggingFace قابلاً للتنزيل من أي مكان في العالم. تحتاج فقط رمز API بصلاحية كتابة
> من `hf.co/settings/tokens` تضعه في الخلية 2 ثم تشغّل الخلية 20.

ما يصل إلى Drive **أوتوماتيكياً** أثناء التدريب (مراقب الخلية 14): آخر نقطتين `DiT_epoch_*.pth` +
`train_console.log` + `loss_curve.png` + `ft_model.pth` عند الاكتمال — كل ذلك في
`MyDrive/rafidain_seedvc/runs/<اسم-التشغيل>/`، بمساحة لا تتجاوز ~2GB.

</div>

In [ ]:
# ============================================================ #
#  الخلية ٢٠ — الرفع الدائم إلى HuggingFace Hub                #
#  النموذج يُخزَّن في حسابك أنت، مجاناً، بلا علاقة بـ Google     #
#  (أعد تشغيلها في أي وقت مستقبلاً — ولو بعد شهور)              #
# ============================================================ #
import sys
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
HF_SCRIPT = ROOT / "hf_upload.py"
HF_SCRIPT.write_text(r'''
import os, time
from huggingface_hub import HfApi

token = os.environ["HF_TOKEN"]
export_dir = os.environ["NB_EXPORT_DIR"]
api = HfApi(token=token)
user = api.whoami()["name"]
repo_id = os.environ.get("NB_REPO_ID", "").strip() or f"{user}/rafidain-seedvc-iraqi"
private = os.environ.get("NB_PRIVATE", "1") == "1"
api.create_repo(repo_id, private=private, exist_ok=True)
print(f"[hf] upload {export_dir} -> {repo_id} (private = {private}) ...")
api.upload_folder(
    folder_path=export_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message=f"Rafidain Iraqi fine-tune ({time.strftime('%Y-%m-%d %H:%M')})",
)
print("[hf] REPO_URL: https://huggingface.co/" + repo_id)
''', encoding="utf-8")

if not CFG["HF_TOKEN"]:
    print("لا يوجد رمز HuggingFace بعد — لا مشكلة: كل أصولك محفوظة على Drive،")
    print("ويمكنك تنفيذ الرفع الدائم في أي وقت لاحق (الخلية تعمل مستقلة):\n")
    print("  1) أنشئ حساباً مجانياً في huggingface.co إن لم يكن لديك")
    print("  2) من hf.co/settings/tokens أنشئ New Token بصلاحية Write وانسخه")
    print('  3) في الخلية 2 اضبط:  CFG["HF_TOKEN"] = "hf_ضع-الرمز-هنا"  ثم أعد تشغيلها')
    print("  4) أعد تشغيل هذه الخلية — سترفع الحزمة كاملة إلى مستودع خاص في حسابك")
else:
    assert (EXPORT_DIR / "seed_uvit_whisper_base_f0_44k_iraqi.pth").exists(), "شغّل خلية التصدير (18) أولاً"
    env = dict(ENV)
    env["HF_TOKEN"] = str(CFG["HF_TOKEN"])
    env["NB_EXPORT_DIR"] = str(EXPORT_DIR)
    env["NB_REPO_ID"] = str(CFG.get("HF_REPO_ID", ""))
    env["NB_PRIVATE"] = "1" if CFG["HF_PRIVATE"] else "0"
    sh([sys.executable, HF_SCRIPT], env=env)
    print('''
[تم] النموذج الآن في حسابك على HuggingFace — بشكل دائم.
لتنزيله في أي وقت مستقبلاً من أي جهاز (بلا Google ولا Colab):
    pip install -U "huggingface_hub[cli]"
    hf download <اسم-حسابك>/rafidain-seedvc-iraqi --local-dir ./mymodel
أو من بايثون:
    from huggingface_hub import snapshot_download
    snapshot_download("<اسم-حسابك>/rafidain-seedvc-iraqi", local_dir="./mymodel")
أو مباشرة من صفحة المستودع في المتصفح.''')

In [ ]:
# ============================================================ #
#  الخلية ٢١ — (اختيارية) تحرير مساحة القرص المحلي             #
#  تحذف المصادر الخام القابلة لإعادة الجلب فقط:                #
#    raw/yodas_webm + مخابئ parquet — وتبقى بيانات التدريب     #
#    المجهزة وكل النقاط والحزم كما هي                          #
#  ملاحظة: علامات .done تبقى، فلن يُعاد التنزيل تلقائياً؛       #
#  لإعادة الجلب مستقبلاً احذف ملف .done المناسب من raw/.done/  #
# ============================================================ #
import sys
import shutil
from pathlib import Path

UV = globals().get("UV") or shutil.which("uv")

def dir_gb(p: Path) -> float:
    if not p.exists():
        return 0.0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

targets = [ROOT / "raw" / "yodas_webm", ROOT / "raw" / "_parquet", ROOT / "raw" / "_safaa_src"]
print(f"المساحة الحرة الآن: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
for t in targets:
    s = dir_gb(t)
    print(f"  {t.name:<12} {s:6.2f} GB" + ("   (قابل لإعادة الجلب بالخليتين 8 و9)" if s > 0 else "   (فارغ)"))

if CFG["CLEAN_RAW_AFTER"]:
    for t in targets:
        if t.exists():
            shutil.rmtree(t)
            print(f"  حُذف: {t.name}")
    if UV:
        sh([UV, "cache", "clean"], check=False)
    else:
        sh([sys.executable, "-m", "pip", "cache", "purge"], check=False)
    print(f"المساحة الحرة بعد التنظيف: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
    print("بيانات التدريب الجاهزة في data/iraqi_finetune — لا حاجة للمصادر الخام إلا لإعادة التجهيز.")
else:
    print('\nللتفعيل: اجعل  CFG["CLEAN_RAW_AFTER"] = True  في الخلية 2 ثم أعد تشغيل هذه الخلية.')

In [ ]:
# ============================================================ #
#  الخلية ٢٢ — النشر: كيف تستخدم النموذج المضبوط؟              #
#  (خلية إرشادية تطبع الأوامر الجاهزة بمساراتك الفعلية)        #
# ============================================================ #
import sys
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
W = EXPORT_DIR / "seed_uvit_whisper_base_f0_44k_iraqi.pth"
C = EXPORT_DIR / "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml"

if not W.exists():
    print("شغّل خلية التصدير (18) أولاً حتى تتوفر الحزمة.")
else:
    print(f'''
==========================================================
 نشر النموذج المضبوط — ثلاثة مسارات جاهزة
==========================================================

(أ) استدلال فوري في هذه الجلسة (البيئة جاهزة الآن):
    cd {SEEDVC}
    {sys.executable} inference.py \\
        --checkpoint {W} \\
        --config     {C} \\
        --source <صوت-المتكلم-الأصلي.wav> \\
        --target  <صوت-مرجعي-عراقي.wav> \\
        --f0-condition true --auto-f0-adjust true \\
        --output  ./out

(ب) خدمة رافِدَين الكاملة على نفس جلسة Colab — REST + WebSocket + واجهة ويب
    بنفق عام مجاني (trycloudflare) للاستنساخ اللحظي من أي مكان.
    المستودع منسوخ هنا: {RAFIDAIN}
    في خلية جديدة الصق ما يلي (توجيه الخدمة إلى أوزانك المضبوطة):

      import os
      os.environ["RAFIDAIN__ENGINE__LOCAL_CHECKPOINT"] = r"{W}"
      os.environ["RAFIDAIN__ENGINE__LOCAL_CONFIG"]     = r"{C}"
      os.environ["RAFIDAIN__ENGINE__DEVICE"]           = "cuda"
      os.environ["RAFIDAIN__API_KEYS"]                 = r'["<مفتاح-API-عشوائي-خاص-بك>"]'
      get_ipython().system("bash {RAFIDAIN}/scripts/install/setup_colab.sh")

    بعد دقائق يطبع رابطاً عاماً مؤقتاً:
      لوحة التحكم:  <الرابط>/web   |   البث الحي:  <الرابط>/live   |   API:  <الرابط>/docs
    (متغيرا البيئة LOCAL_CHECKPOINT / LOCAL_CONFIG هما المساران أعلاه — يُقرآن مباشرة من البيئة)

(ج) على أي جهاز GPU آخر (خادمك / RunPod / غيره):
    1) نزّل الحزمة من Drive (rafidain_seedvc/exports) أو من HuggingFace (الخلية 20)
    2) git clone https://github.com/Plachtaa/seed-vc && cd seed-vc
    3) python inference.py --checkpoint <الحزمة>/seed_uvit_whisper_base_f0_44k_iraqi.pth \\
           --config <الحزمة>/config_dit_mel_seed_uvit_whisper_base_f0_44k.yml \\
           --f0-condition true --auto-f0-adjust true \\
           --source a.wav --target ref.wav --output ./out
    أو خدمة رافِدَين الكاملة بنفس متغيرَي البيئة في (ب) — الدليل: DEPLOYMENT.md و WEBUI.md

تذكير أخلاقي: استنساخ صوت شخص حقيقي يتطلب موافقته الصريحة الموثقة،
وأفصح للطرف الآخر عند ما يوجبه القانون — راجع قسم الأمان في README المستودع.
''')

<div dir="rtl">

## استكشاف الأخطاء

| المشكلة | الحل |
|---|---|
| انقطعت الجلسة / أُعيد التشغيل | افتح الدفتر ← **Run all**: البيئة تُبنى، الخلية 13 تستعيد النقاط من Drive، التدريب يكمل من آخر نقطة (الخسارة ≤ آخر فترة حفظ). البيانات الخام إن فُقدت تُجلب مجدداً بالخليتين 8-9 (استئناف كامل) |
| «No GPU available» | Runtime ← Change runtime type ← Hardware accelerator: GPU ← Save، ثم أعد تشغيل الخلايا بالترتيب |
| `CUDA out of memory` أثناء التدريب | خفّض الدفعة: `CFG["BATCH"] = 1` في الخلية 2، أعد تشغيلها ثم أعد الخلية 14 — يستأنف من آخر نقطة |
| انقطعت الجلسة وأنا بعيد عن المتصفح | طبيعي بعد ~90 دقيقة خمول — الخلية 14 النشطة (التدريب داخل النواة) تمنع ذلك أثناء عملها؛ نقاطك على Drive على أي حال |
| نفدت مساحة Drive | المراقب يُبقي آخر نقطتين فقط (~1GB)؛ احذف مجلدات runs قديمة من Drive، أو اعتمد على HF Hub (الخلية 20) كطبقة وحيدة |
| نفد القرص المحلي `/content` | شغّل خلية التنظيف (21)، أو قلّل `YODAS_HOURS` وأعد الخليتين 9 و11 (الموجود يُتخطى) |
| Rate limit من HuggingFace (429) | Xet معطّل أصلاً هنا؛ انتظر دقائق وأعد المحاولة، أو اضبط `CFG["HF_ENDPOINT"] = "https://hf-mirror.com"` في الخلية 2 |
| بطء تنزيل YODAS | ارفع `FETCH_WORKERS` إلى 32 في الخلية 2 ثم أعد الخلية 9 — يستأنف دون إعادة ما نُزّل |
| `too few training clips` في الخلية 11 | تأكد أن الخليتين 8 و9 أنزلتا بيانات (انظر مجلد `raw/`) ثم أعد المحاولة |
| فشلت خلية التحقق (17) | غير قاتلة — الأوزان تصدَّر على أي حال؛ جرّب أمر inference.py اليدوي المطبوع في الخلية 22 |
| فشل استيراد torch داخل النواة | أعد تشغيل الخلية 5 (تثبيت داخل النواة عبر uv — دقائق معدودة) ثم الخلية 14 |
| أعدت تشغيل النواة وضاعت المتغيرات | شغّل الخلايا بالترتيب من 1 — الإعدادات محفوظة (على Drive إن كان موصولاً) وتعود كما كانت |
| أريد تجربة نقطة وسيطة | ملفات `DiT_epoch_*_step_*.pth` (محلياً أو على Drive) تُقبل مباشرة في `inference.py --checkpoint` |

</div>

<div dir="rtl">

## الخلاصة — أين كل شيء؟ (خريطة المقتنيات الدائمة)

| الأصل | المسار الدائم |
|---|---|
| حزمة النموذج النهائية | **Google Drive:** `MyDrive/rafidain_seedvc/exports/<اسم-التشغيل>/` |
| أرشيف النموذج (ملف واحد) | **Google Drive:** `MyDrive/rafidain_seedvc/exports/rafidain_seedvc_<اسم-التشغيل>.tar.gz` |
| نسخة HuggingFace الدائمة | `https://huggingface.co/<حسابك>/rafidain-seedvc-iraqi` (الخلية 20) |
| نقاط التدريب الاحتياطية + السجلات + منحنى الخسارة | **Google Drive:** `MyDrive/rafidain_seedvc/runs/<اسم-التشغيل>/` (تحديث تلقائي أثناء التدريب) |
| الإعدادات | **Google Drive:** `MyDrive/rafidain_seedvc/nb_config.json` |
| بيانات التدريب المجهزة | محلي `/content/iraqi_seedvc/data/iraqi_finetune/` — تُبنى من جديد عند الحاجة (استئناف كامل) |
| البيئة + المستودعات + الخام | محلي `/content/iraqi_seedvc/` — تُبنى من جديد بالخلايا 4-9 |

**لتنزيل النموذج في أي وقت مستقبلاً** — الطبقة الأضمن (HuggingFace، بلا Google نهائياً):

```bash
pip install -U "huggingface_hub[cli]"
hf download <اسم-حسابك>/rafidain-seedvc-iraqi --local-dir ./mymodel
```

ثم تشغيله فوراً على أي جهاز فيه GPU:

```bash
git clone https://github.com/Plachtaa/seed-vc && cd seed-vc
python inference.py \
    --checkpoint ./mymodel/seed_uvit_whisper_base_f0_44k_iraqi.pth \
    --config     ./mymodel/config_dit_mel_seed_uvit_whisper_base_f0_44k.yml \
    --source a.wav --target ref.wav --output ./out
```

أو من Drive مباشرة: نزّل مجلد `rafidain_seedvc/exports/<اسم-التشغيل>/` — فيه نفس الملفات
مع README أوامر الاستخدام و`training_report.json` التوثيقي.

وللاستنساخ اللحظي الكامل (REST + WebSocket + واجهة ويب بنفق عام): خلية النشر (22) المسار (ب)
تُشغّل خدمة رافِدَين كاملة على نفس جلسة Colab بأوزانك المضبوطة خلال دقائق.

</div>